<a href="https://colab.research.google.com/github/Zanz-19/Modulo-6/blob/master/Proyecto_Vison_Audio/M6_Proyecto_LOS_LINUXEROS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TAE-IA · Módulo 6 — Proyecto final: Sistema cibernético de transducción de ideas: generación visual, verificación automática e informe auditivo bilingüe.

**Equipo:** José Ramón Sánchez Acosta · Josué Alejandro Murillo Orozco

**Idea.** Describes una imagen con tu voz (en español o inglés). La app la dibuja, comprueba si el resultado se parece a lo que pediste y te lo cuenta en voz alta, en español y en inglés.

`voz → Whisper → (traducción) → Stable Diffusion → imagen → YOLOv8 + BLIP + OCR + CLIP → veredicto → informe ES/EN → Piper (voz)`

| Requisito del proyecto | Cómo se cumple |
|---|---|
| Al menos 1 modelo de visión | YOLOv8n (detección), BLIP-large (caption), Tesseract (OCR), CLIP ViT-B/32 (similitud imagen-texto) y Stable Diffusion 1.5 (generación) |
| Al menos 1 modelo de audio | Whisper small (entrada: voz a texto) y Piper (salida: voz del informe); los dos se ven en la interfaz |
| App Gradio en vivo | Sección 10: `launch()` con enlace público |

**Cómo usarlo**
1. Runtime con GPU T4 y conexión a internet (se descargan los modelos).
2. Ejecutar todas las celdas en orden. La primera vez descarga varios GB de modelos, así que tarda varios minutos.
3. Abrir el enlace público `*.gradio.live` en una **pestaña nueva**: el micrófono no funciona dentro del recuadro de Colab. Con Brave, desactivar los Shields para esa página.
4. Grabar una idea, subir un audio o usar los ejemplos, y pulsar **Crear imagen**.

No usa Drive ni archivos de apoyo.

## 1. Instalación (versiones fijas)
Instala Tesseract (OCR, con español) y las librerías con la versión exacta con la que se probó el notebook. `ffmpeg` ya viene en Colab; Whisper lo usa para leer audio.

In [1]:
!apt-get install -y -q tesseract-ocr tesseract-ocr-spa > /dev/null
!pip install -q ultralytics==8.4.170 transformers==5.16.1 accelerate==1.14.0 gradio==6.26.0 pytesseract==0.3.13 openai-whisper==20250625 piper-tts==1.8.0 diffusers==0.40.0 sentencepiece==0.2.2
!ffmpeg -version | head -n 1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 57.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 10.1 MB/s eta 0:00:00
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


## 2. Configuración
Carpeta de pesos en el disco del runtime, semillas fijas (42), verificación de GPU y la función `vram()` para medir memoria en cada etapa. No usa Google Drive.

In [2]:
import os, sys, time, random
from typing import NamedTuple
import numpy as np
import torch

# Los pesos van al disco del runtime (rápido, sin el problema de symlinks de Drive)
MODEL_CACHE = '/content/models'
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['HF_HOME']         = MODEL_CACHE
os.environ['TORCH_HOME']      = MODEL_CACHE
os.environ['YOLO_CONFIG_DIR'] = MODEL_CACHE     # antes de importar ultralytics

if not torch.cuda.is_available():
    raise RuntimeError('Sin GPU. Entorno de ejecución > Cambiar tipo > T4 GPU')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def vram(tag=''):
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM {used:5.2f} / {total:.1f} GB   {tag}')

print(torch.cuda.get_device_name(0)); vram('vacía')
print('Python', sys.version.split()[0], '| torch', torch.__version__)

Tesla T4
  VRAM  0.00 / 15.6 GB   vacía
Python 3.13.15 | torch 2.11.0+cu128


## 3. Audio de entrada: Whisper
`coerce_audio` valida lo que llega (que exista, se lea, dure entre 1 y 15 s y no sea silencio) y lanza `EntradaInvalida` con un mensaje para el usuario. `transcribe_idea` detecta español o inglés y transcribe con Whisper *small*, descartando los segmentos sin voz (`no_speech_prob`). Devuelve `Heard(texto, idioma, probabilidad, duración)`.

In [3]:
import whisper

class EntradaInvalida(ValueError):
    """Error causado por la entrada del usuario; su mensaje ya está escrito para mostrarse."""

SR            = whisper.audio.SAMPLE_RATE     # 16000 Hz: lo que Whisper espera
MIN_SEC, MAX_SEC = 1.0, 15.0                  # duración válida de la nota de voz
MIN_RMS       = 0.005                         # energía mínima; el ruido lo descarta el filtro no_speech_prob
NO_SPEECH_MAX = 0.6                           # mismo umbral de L20 para descartar segmentos sin voz
LANGS         = ('es', 'en')                  # idiomas que soporta la app

asr = whisper.load_model('small', download_root=f'{MODEL_CACHE}/whisper')   # queda en la GPU
vram('tras Whisper small')

class Heard(NamedTuple):
    text: str        # lo que se entendió
    lang: str        # 'es' o 'en'
    lang_p: float    # probabilidad de ese idioma (1.0 si el usuario lo forzó)
    seconds: float   # duración del audio

def coerce_audio(x):
    """Ruta de archivo -> np.float32 mono 16 kHz, o EntradaInvalida con mensaje para el usuario."""
    if x is None or (isinstance(x, str) and not x):
        raise EntradaInvalida('No se recibió ningún audio. Graba o sube uno.')
    if not isinstance(x, (str, os.PathLike)):
        raise EntradaInvalida('Entrada no soportada. Graba con el micrófono o sube un archivo de audio.')
    try:
        y = whisper.load_audio(str(x))        # ffmpeg decodifica wav/mp3/m4a/webm y remuestrea a 16 kHz mono
    except Exception:
        raise EntradaInvalida('No pude leer ese archivo como audio. Prueba con .wav, .mp3 o el micrófono.') from None
    dur = len(y) / SR
    if dur < MIN_SEC:
        raise EntradaInvalida(f'El audio es demasiado corto ({dur:.1f} s). Mínimo {MIN_SEC:.0f} s.')
    if dur > MAX_SEC:
        raise EntradaInvalida(f'El audio es demasiado largo ({dur:.0f} s). Máximo {MAX_SEC:.0f} s: describe tu idea en pocas frases.')
    rms = float(np.sqrt(np.mean(y ** 2)))     # energía media de la señal
    if rms < MIN_RMS:
        raise EntradaInvalida('El audio está casi en silencio. Acércate al micrófono y vuelve a intentar.')
    return y

def detect_lang(y):
    """Whisper elige entre ~100 idiomas; aquí solo comparamos es vs en."""
    mel = whisper.log_mel_spectrogram(whisper.pad_or_trim(y), asr.dims.n_mels)   # espectrograma de 30 s
    mel = mel.to(asr.device, next(asr.parameters()).dtype)                        # mismo dtype que el modelo
    _, probs = asr.detect_language(mel)                                           # dict {idioma: prob}
    lang = max(LANGS, key=lambda l: probs[l])
    return lang, float(probs[lang])

def transcribe_idea(y, language=None):
    lang, p = (language, 1.0) if language in LANGS else detect_lang(y)
    res  = asr.transcribe(y, language=lang, fp16=True)
    segs = [s for s in res['segments'] if s.get('no_speech_prob', 0) <= NO_SPEECH_MAX]   # filtro de L20
    text = ' '.join(s['text'].strip() for s in segs).strip()
    if not text:
        raise EntradaInvalida('No se entendió ninguna palabra. Habla más cerca del micrófono y vuelve a intentar.')
    return text, lang, p

def hear(x, language=None):
    """Etapa 1 completa: entrada cruda -> Heard, o EntradaInvalida."""
    y = coerce_audio(x)
    text, lang, p = transcribe_idea(y, language)
    return Heard(text, lang, p, len(y) / SR)

100%|███████████████████████████████████████| 461M/461M [00:07<00:00, 68.4MiB/s]


  VRAM  0.97 / 15.6 GB   tras Whisper small


## 4. Voces del informe (Piper)
Dos voces locales de Piper (español de México e inglés de EE. UU.). `piper_wav` convierte texto en un archivo .wav. También se generan dos audios de ejemplo que la interfaz ofrece como respaldo si falla el micrófono.

In [4]:
import wave
from huggingface_hub import hf_hub_download
from piper import PiperVoice, SynthesisConfig

def load_piper(stem):
    """Descarga (si falta) una voz de Piper y la carga."""
    onnx = hf_hub_download('rhasspy/piper-voices', stem + '.onnx', local_dir=f'{MODEL_CACHE}/piper')
    hf_hub_download('rhasspy/piper-voices', stem + '.onnx.json', local_dir=f'{MODEL_CACHE}/piper')
    return PiperVoice.load(onnx)

def piper_wav(voice, text, path):
    """Texto -> archivo .wav con esa voz."""
    with wave.open(path, 'wb') as wf:
        voice.synthesize_wav(text, wf, syn_config=SynthesisConfig())
    return path

voices = {'es': load_piper('es/es_MX/claude/high/es_MX-claude-high'),
          'en': load_piper('en/en_US/lessac/medium/en_US-lessac-medium')}

# Audios de ejemplo para la interfaz (respaldo si falla el micrófono)
EXAMPLES_DIR = '/content/ejemplos'; os.makedirs(EXAMPLES_DIR, exist_ok=True)
clips = {
    'es_valido': piper_wav(voices['es'], 'Un gato naranja dormido sobre un sofá azul, junto a una ventana con lluvia.', f'{EXAMPLES_DIR}/es.wav'),
    'en_valido': piper_wav(voices['en'], 'An orange cat sleeping on a blue sofa, next to a rainy window.', f'{EXAMPLES_DIR}/en.wav'),
}

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


es/es_MX/claude/high/es_MX-claude-high.o(…): reconstructing file:   0%|          |  0.00B / 63.1MB            

es/es_MX/claude/high/es_MX-claude-high.o(…): downloading bytes:           |  0.00B            

es_MX-claude-high.onnx.json:   0%|          | 0.00/4.96k [00:00<?, ?B/s]

en/en_US/lessac/medium/en_US-lessac-medi(…): reconstructing file:   0%|          |  0.00B / 63.2MB            

en/en_US/lessac/medium/en_US-lessac-medi(…): downloading bytes:           |  0.00B            

en_US-lessac-medium.onnx.json:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

## 5. Del prompt a la imagen: Stable Diffusion 1.5
`draw(prompt)` genera una imagen de 512×512 con 25 pasos y guidance 7.5 (en el L01, más de 25 pasos casi no mejoraba la imagen y duplicaba el tiempo). Mantiene activo el filtro de seguridad: si bloquea una imagen, lanza `ImagenBloqueada` con un mensaje. Avisa si el prompt supera los 77 tokens que lee CLIP.

In [5]:
from diffusers import StableDiffusionPipeline
from PIL import Image

SD_ID = 'runwayml/stable-diffusion-v1-5'     # el mismo de L01
STEPS, CFG = 25, 7.5          # L01 §2.3-2.4: más de 25 pasos casi no mejora y cuesta el doble

pipe = StableDiffusionPipeline.from_pretrained(
    SD_ID, dtype=torch.float16, cache_dir=MODEL_CACHE).to('cuda')   # fp16 para que quepa holgado en la T4
pipe.set_progress_bar_config(disable=True)       # quita la barra tqdm; el progreso lo mostrará Gradio
assert pipe.safety_checker is not None, 'El filtro de seguridad debe estar activo en la demo pública'
vram('tras Stable Diffusion')

class ImagenBloqueada(EntradaInvalida):
    """El filtro de seguridad bloqueó la imagen; el mensaje ya está escrito para el usuario."""

class Picture(NamedTuple):
    image: Image.Image
    seed: int          # con la misma semilla y prompt sale la misma imagen
    seconds: float
    truncated: bool    # el prompt pasó de 77 tokens y CLIP cortó el final

def _generate(prompt, gen):                      # aparte, para poder sustituirla en las pruebas
    return pipe(prompt, num_inference_steps=STEPS, guidance_scale=CFG, generator=gen)

def draw(prompt, seed=None):
    """Prompt en inglés -> Picture, o EntradaInvalida / ImagenBloqueada con mensaje para el usuario."""
    if not isinstance(prompt, str) or not prompt.strip():
        raise EntradaInvalida('No hay ninguna descripción para dibujar.')
    prompt = prompt.strip()
    truncated = len(pipe.tokenizer(prompt).input_ids) > pipe.tokenizer.model_max_length   # 77 tokens
    seed = int(np.random.default_rng().integers(2**31)) if seed is None else int(seed)    # semilla nueva por pedido
    gen = torch.Generator('cuda').manual_seed(seed)
    t0 = time.time()
    res = _generate(prompt, gen)
    flags = res.nsfw_content_detected
    if flags and flags[0]:                       # el pipeline devuelve imagen negra y esta marca
        raise ImagenBloqueada('El filtro de seguridad bloqueó la imagen. Describe otra escena.')
    return Picture(res.images[0], seed, time.time() - t0, truncated)

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

  VRAM  3.74 / 15.6 GB   tras Stable Diffusion


## 6. Verificación de la imagen
Tres señales independientes: el pipeline del L12 (YOLO, BLIP, OCR), CLIP y el cruce del prompt con las clases COCO.

### 6.1 Pipeline del L12
YOLOv8n detecta; `plan()` decide a qué modelo va cada recorte (seres vivos → BLIP-large para el caption; objetos con texto → Tesseract OCR; el resto solo etiqueta; si no detecta nada, caption de la imagen completa); `run_tasks()` ejecuta cada tarea sin que un fallo tumbe a las demás.

In [6]:
from PIL import Image, ImageOps, ImageDraw


# ---------- L12: YOLO y detect ----------
from ultralytics import YOLO
yolo = YOLO('yolov8n.pt')           # 6 MB; el más rápido y el menos preciso de la familia
print(f'YOLO cargado: {len(yolo.names)} clases COCO'); vram('tras YOLO')

class Det(NamedTuple):
    label: str
    conf: float
    box: tuple          # (x1, y1, x2, y2) en píxeles

def detect(img, conf=0.25):
    """PIL RGB -> [Det], la más segura primero."""
    # Se pasa el PIL directo: ultralytics lo trata como RGB. Un np.array se interpreta
    # como BGR (convención OpenCV) y cambiaría los colores que ve el modelo.
    res = yolo(img, conf=conf, verbose=False)[0]
    dets = [Det(res.names[int(b.cls)], float(b.conf), tuple(map(int, b.xyxy[0].tolist())))
            for b in res.boxes]
    return sorted(dets, key=lambda d: -d.conf)

# ---------- L12: enrutamiento ----------
LIVING = {'person', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe'}
TEXTY  = {'stop sign', 'book', 'tv', 'laptop', 'cell phone', 'bottle', 'parking meter'}
ROUTES = {**{c: 'caption' for c in LIVING}, **{c: 'ocr' for c in TEXTY}}

# Un typo en una clase ('cellphone') nunca fallaría: simplemente nunca enrutaría. Se verifica al cargar.
_bad = sorted(c for c in ROUTES if c not in set(yolo.names.values()))
assert not _bad, f'Clases que no existen en COCO: {_bad}'

MAX_OBJECTS = 6      # máximo de detecciones que se procesan por imagen
MIN_CROP    = 24     # px; debajo de esto el recorte no sirve para BLIP ni para OCR
PAD_FRAC    = 0.08   # margen alrededor de la caja (proporcional a su tamaño)

class Task(NamedTuple):
    label: str
    conf: float
    box: tuple
    route: str          # 'caption' | 'ocr' | 'skip'
    crop: Image.Image
    note: str

def crop_box(img, box, pad_frac=PAD_FRAC):
    """Recorta con margen; el margen se recorta a los límites de la imagen."""
    x1, y1, x2, y2 = box
    pad = int(pad_frac * max(x2 - x1, y2 - y1))
    w, h = img.size
    return img.crop((max(0, x1 - pad), max(0, y1 - pad), min(w, x2 + pad), min(h, y2 + pad)))

def _inter_frac(a, b):
    """Fracción del área de la caja a que cae dentro de la caja b."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area  = (a[2] - a[0]) * (a[3] - a[1])
    return inter / area if area else 0.0

def drop_nested(dets, thr=0.85):
    """Quita una detección si otra de la MISMA clase y más segura la contiene casi por completo."""
    kept = []
    for d in dets:                       # ya vienen ordenadas de mayor a menor confianza
        if any(k.label == d.label and _inter_frac(d.box, k.box) >= thr for k in kept):
            continue
        kept.append(d)
    return kept

def plan(img, dets, max_objects=MAX_OBJECTS):
    """(PIL, [Det]) -> [Task]. No ejecuta ningún modelo: solo decide a dónde va cada recorte."""
    dets = drop_nested(dets)

    w, h = img.size
    if not dets:
        # Encontrar nada es una ENTRADA VÁLIDA (COCO solo tiene 80 clases), no un error.
        return [Task('(imagen completa)', 0.0, (0, 0, w, h), 'caption', img,
                     'YOLO no encontró objetos: se describe la imagen completa')]
    tasks = []
    for d in dets[:max_objects]:
        crop  = crop_box(img, d.box)
        route = ROUTES.get(d.label, 'skip')
        note  = '' if route != 'skip' else 'sin ruta para esta clase'
        if crop.width < MIN_CROP or crop.height < MIN_CROP:
            route, note = 'skip', f'recorte demasiado pequeño ({crop.width}x{crop.height})'
        tasks.append(Task(d.label, d.conf, d.box, route, crop, note))
    return tasks

ROUTE_COLOR = {'caption': '#22aa55', 'ocr': '#2266dd', 'skip': '#999999'}

def draw_tasks(img, tasks):
    """Devuelve una copia con cada caja coloreada según su ruta."""
    out = img.copy(); dr = ImageDraw.Draw(out)
    for t in tasks:
        if t.label == '(imagen completa)':
            continue
        dr.rectangle(t.box, outline=ROUTE_COLOR[t.route], width=3)
        dr.text((t.box[0] + 4, t.box[1] + 2), f'{t.label} {t.conf:.2f} -> {t.route}', fill=ROUTE_COLOR[t.route])
    return out

# ---------- L12: BLIP-large ----------
from transformers import BlipProcessor, BlipForConditionalGeneration
import re

BLIP_ID    = 'Salesforce/blip-image-captioning-large'
blip_proc  = BlipProcessor.from_pretrained(BLIP_ID)
blip_model = BlipForConditionalGeneration.from_pretrained(
    BLIP_ID, dtype=torch.float16).to('cuda').eval()
vram('tras BLIP')

def clean_caption(s):
    s = re.sub(r"\s+'\s*(s|t|re|ve|ll|d|m)\b", r"'\1", s.strip())   # "it ' s" -> "it's"
    s = re.sub(r'^there (is|are) ', '', s, flags=re.I)                # quita "there is/are"
    return s[:1].upper() + s[1:]

def caption_crop(img, max_new_tokens=40):
    """PIL RGB -> str. float16 a la entrada, igual que como se cargó el modelo."""
    inp = blip_proc(images=img, return_tensors='pt').to('cuda', torch.float16)
    with torch.no_grad():
        out = blip_model.generate(**inp, max_new_tokens=max_new_tokens, num_beams=3)
    return clean_caption(blip_proc.decode(out[0], skip_special_tokens=True))

# ---------- L12: OCR ----------
import cv2, pytesseract

OCR_LANG, OCR_TARGET_H, MAX_INK_FRAC = 'eng+spa', 400, 0.15

def _accept(w, c):
    """Palabra confiable: >=3 caracteres con conf>=70, o cualquier longitud con conf>=85."""
    n = sum(ch.isalnum() for ch in w)
    return n > 0 and ((n >= 3 and c >= 70) or c >= 85)

def _ocr_pass(arr):
    d = pytesseract.image_to_data(arr, lang=OCR_LANG, config='--psm 11 --oem 3',
                                  output_type=pytesseract.Output.DICT)
    out = []
    for w, c in zip(d['text'], d['conf']):
        w = w.strip()
        if w and _accept(w, float(c)):
            out.append((w, float(c)))
    return out

def _sin_gigantes(chan):
    """Binariza (Otsu) y borra las manchas enormes (aro, poste, fondo). Tinta negra sobre blanco."""
    _, bw = cv2.threshold(chan, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    outs = []
    for fg in (bw, 255 - bw):                      # probar ambas polaridades
        n, lab, st, _ = cv2.connectedComponentsWithStats(fg, connectivity=8)
        keep = np.zeros_like(fg)
        for i in range(1, n):
            if st[i, cv2.CC_STAT_AREA] <= MAX_INK_FRAC * fg.size:
                keep[lab == i] = 255
        outs.append(255 - keep)
    return outs

def ocr_variants(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    v = {'gris': gray, 'gris inv': 255 - gray}
    for ch_name, ch in (('gris', gray), ('min(RGB)', rgb.min(axis=2))):
        for k, a in enumerate(_sin_gigantes(ch)):
            v[f'{ch_name} bin{k}'] = a
    return v

def ocr_crop(img):
    """PIL RGB -> (texto, confianza media, variante ganadora). ('', 0.0, '') = no hay texto legible."""
    rgb = np.array(img.convert('RGB'))
    if rgb.shape[0] < OCR_TARGET_H:                # recortes chicos: subir resolución
        f = OCR_TARGET_H / rgb.shape[0]
        rgb = cv2.resize(rgb, None, fx=f, fy=f, interpolation=cv2.INTER_CUBIC)
    best, best_name = [], ''
    for name, arr in ocr_variants(rgb).items():
        words = _ocr_pass(arr)
        if sum(c for _, c in words) > sum(c for _, c in best):
            best, best_name = words, name
    if not best:
        return '', 0.0, ''
    return ' '.join(w for w, _ in best), sum(c for _, c in best) / len(best), best_name

# ---------- L12: ejecutar tareas ----------
class Result(NamedTuple):
    task: Task
    output: str      # caption, texto OCR, o '' si la ruta fue skip
    info: str        # tiempo, confianza o motivo

def run_tasks(tasks):
    """[Task] -> [Result]. Un recorte que falle no tumba a los demás."""
    results = []
    for t in tasks:
        t0 = time.time()
        try:
            if t.route == 'caption':
                out, info = caption_crop(t.crop), f'BLIP {time.time()-t0:.2f}s'
            elif t.route == 'ocr':
                text, conf, var = ocr_crop(t.crop)
                out  = text if text else '(sin texto legible)'
                info = f'OCR conf {conf:.0f} [{var}], {time.time()-t0:.2f}s' if text else 'OCR sin texto confiable'
            else:
                out, info = '', t.note
        except Exception as e:
            print(f'  [fallo en {t.label}] {type(e).__name__}: {e}')
            out, info = '(error al procesar este objeto)', 'falló'
        results.append(Result(t, out, info))
    return results

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/content/models/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLO cargado: 80 clases COCO
  VRAM  3.74 / 15.6 GB   tras YOLO


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.60k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.88GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

  VRAM  4.65 / 15.6 GB   tras BLIP


### 6.2 CLIP y cruce con las clases COCO
CLIP mide qué tan parecidos son la imagen y el prompt (similitud coseno). `requested_classes()` busca en el prompt las clases COCO que pidió (con sinónimos como sofa→couch; "orange" solo cuenta como fruta en plural) y `coverage()` comprueba si YOLO las vio.

In [7]:
from transformers import CLIPModel, CLIPProcessor

# ---------- Señal 1: CLIP (similitud imagen-prompt) ----------
CLIP_ID    = 'openai/clip-vit-base-patch32'          # el mismo de L10
clip_proc  = CLIPProcessor.from_pretrained(CLIP_ID, cache_dir=MODEL_CACHE)
clip_model = CLIPModel.from_pretrained(CLIP_ID, cache_dir=MODEL_CACHE).to('cuda').eval()
vram('tras CLIP')

def clip_matrix(imgs, texts):
    """[PIL] x [str] -> array [i, j] = similitud coseno de la imagen i con el texto j."""
    inp = clip_proc(text=texts, images=imgs, return_tensors='pt', padding=True, truncation=True).to('cuda')
    with torch.no_grad():
        logits = clip_model(**inp).logits_per_image          # = logit_scale * coseno
        cos    = logits / clip_model.logit_scale.exp()       # quita la escala -> coseno puro
    return cos.cpu().numpy()

def clip_score(img, text):
    return float(clip_matrix([img], [text])[0, 0])

# ---------- Señal 2: ¿YOLO vio lo que pidió el prompt? ----------
COCO_NAMES = [n for n in yolo.names.values() if n != 'orange']   # 'orange' casi siempre es un COLOR: se trata aparte
SYNONYMS = {
    'sofa': 'couch', 'bike': 'bicycle', 'motorbike': 'motorcycle', 'plane': 'airplane',
    'television': 'tv', 'monitor': 'tv', 'phone': 'cell phone', 'smartphone': 'cell phone',
    'fridge': 'refrigerator', 'table': 'dining table', 'plant': 'potted plant', 'ship': 'boat',
    'puppy': 'dog', 'kitten': 'cat',
    **{w: 'person' for w in ('man', 'men', 'woman', 'women', 'people', 'boy', 'girl', 'child',
                              'children', 'kid', 'baby', 'astronaut')},
}
_bad = sorted(set(SYNONYMS.values()) - set(yolo.names.values()))     # mismo seguro que en L12 contra typos
assert not _bad, f'Sinónimos hacia clases que no existen en COCO: {_bad}'
VOCAB = {**{n: n for n in COCO_NAMES}, **SYNONYMS, 'orangefruit': 'orange'}

def requested_classes(prompt):
    """Prompt en inglés -> clases COCO que pide, en orden de aparición."""
    text = re.sub(r'[^a-z\s]', ' ', prompt.lower())
    # 'orange' es fruta solo en plural ("two oranges") o con 'fruit'; en "an orange cat" es un color
    text = re.sub(r'\boranges?\s+fruit\b|\boranges\b', ' orangefruit ', f' {text} ')
    found = {}
    for word in sorted(VOCAB, key=len, reverse=True):          # lo más largo primero: 'teddy bear' antes que 'bear'
        pat = rf'\b{word}(?:s|es)?\b'                          # acepta plural simple
        for m in re.finditer(pat, text):
            found.setdefault(VOCAB[word], m.start())           # posición de la primera aparición
        text = re.sub(pat, lambda m: ' ' * len(m.group()), text)   # se borra (mismo largo) para no contarla dos veces
    return sorted(found, key=found.get)

class Coverage(NamedTuple):
    requested: list    # clases COCO que pide el prompt
    found: list        # ... que YOLO sí detectó
    missing: list      # ... que YOLO no detectó

def coverage(prompt, dets):
    req  = requested_classes(prompt)
    seen = {d.label for d in dets}
    return Coverage(req, [c for c in req if c in seen], [c for c in req if c not in seen])

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  VRAM  5.25 / 15.6 GB   tras CLIP


### 6.3 `verify()`: el veredicto
| CLIP | ¿Falta algo de lo pedido en YOLO? | Veredicto |
|---|---|---|
| ≥ 0.27 | no | coincide |
| ≥ 0.27 | sí | coincide parcialmente |
| 0.22 – 0.27 | — | dudoso |
| < 0.22 | — | no coincide |

Los umbrales (`CLIP_LOW = 0.22`, `CLIP_HIGH = 0.27`) salen de una calibración con 10 imágenes generadas: los pares correctos puntuaron entre 0.292 y 0.409 y los equivocados entre 0.047 y 0.218.

**Límite medido:** CLIP separa bien escenas distintas, pero no detalles. Con una sola palabra cambiada (gato→perro, bici roja→azul) el margen fue de solo +0.008 a +0.058. Por eso la app avisa que comprueba la escena, no colores ni detalles finos.

In [8]:
CLIP_LOW, CLIP_HIGH = 0.22, 0.27     # calibración con 10 imágenes (ver texto 6.3): equivocados <= 0.218, correctos >= 0.292

class Verification(NamedTuple):
    clip: float
    band: str            # 'alto' | 'medio' | 'bajo', según CLIP
    coverage: Coverage
    tasks: list          # [Task] del L12
    results: list        # [Result] del L12
    verdict: str         # 'coincide' | 'coincide parcialmente' | 'dudoso' | 'no coincide'
    boxes: Image.Image   # la imagen con las cajas de YOLO dibujadas

def clip_band(score):
    if score >= CLIP_HIGH: return 'alto'
    if score >= CLIP_LOW:  return 'medio'
    return 'bajo'

def decide(band, cov):
    if band == 'bajo':  return 'no coincide'
    if band == 'medio': return 'dudoso'
    return 'coincide parcialmente' if cov.missing else 'coincide'

def verify(prompt, img):
    """(prompt EN, PIL) -> Verification. Corre YOLO + BLIP + OCR (L12) y CLIP, y decide."""
    dets    = detect(img)
    cov     = coverage(prompt, dets)
    tasks   = plan(img, dets)
    results = run_tasks(tasks)
    score   = clip_score(img, prompt)
    band    = clip_band(score)
    return Verification(score, band, cov, tasks, results, decide(band, cov), draw_tasks(img, tasks))

## 7. Informe y voz en inglés
`facts_from()` junta los hechos sin idioma (qué se entendió, veredicto, qué se pidió y qué encontró YOLO, dudas, caption, texto del OCR) y `report_en()` los convierte en frases. `speak()` las narra con Piper.

In [9]:
# ---------- Etapa 4a: de la verificación a un informe ----------
def ambiguous(tasks, thr=0.85):
    """Pares de clases DISTINTAS sobre casi la misma caja: YOLO duda entre ellas."""
    real = [t for t in tasks if t.label != '(imagen completa)']
    out = []
    for i, a in enumerate(real):
        for b in real[i + 1:]:
            if (a.label != b.label and _inter_frac(a.box, b.box) >= thr
                    and _inter_frac(b.box, a.box) >= thr):          # cada caja casi dentro de la otra
                out.append((a.label, b.label))
    return out

class Facts(NamedTuple):
    """Los hechos del informe, sin idioma: de aquí salen el texto en inglés y, después, en español."""
    heard: str
    verdict: str
    requested: list
    found: list
    missing: list
    caption: str
    texts: list
    doubts: list

def facts_from(heard_text, v):
    caps = []
    for r in v.results:                                  # sin repetir: gato y perro daban el mismo caption
        if r.task.route == 'caption' and r.output and r.output not in caps:
            caps.append(r.output)
    texts = [r.output for r in v.results
             if r.task.route == 'ocr' and r.output and not r.output.startswith('(')]   # '(sin texto legible)' no cuenta
    req = set(v.coverage.requested)
    doubts = [(a, b) for a, b in ambiguous(v.tasks) if a in req or b in req]          # solo dudas sobre lo pedido
    return Facts(heard_text, v.verdict, v.coverage.requested, v.coverage.found,
                 v.coverage.missing, caps[0] if caps else '', texts, doubts)

VERDICT_EN = {
    'coincide': 'matches your idea',
    'coincide parcialmente': 'partly matches your idea',
    'dudoso': 'may not match your idea',
    'no coincide': 'does not match your idea',
}

def join_en(items):
    items = list(items)
    return items[0] if len(items) == 1 else ', '.join(items[:-1]) + ' and ' + items[-1]

def report_en(f):
    s = [f'I understood: {f.heard.rstrip(".")}.',
         f'The generated image {VERDICT_EN[f.verdict]}.']
    if f.requested:
        if not f.missing:
            s.append(f'The detector found everything you asked for: {join_en(f.found)}.')
        elif f.found:
            s.append(f'The detector found {join_en(f.found)}, but not {join_en(f.missing)}.')
        else:
            s.append(f'The detector did not find {join_en(f.missing)}.')
    for a, b in f.doubts:
        s.append(f'It hesitated between {a} and {b}.')
    if f.caption:
        s.append(f'The vision model describes it as: {f.caption[0].lower() + f.caption[1:].rstrip(".")}.')
    if f.texts:
        s.append(f'Text read in the image: {join_en(f.texts)}.')
    return ' '.join(s)

# ---------- Etapa 4b: voz (las voces de Piper se cargan en la sección 4) ----------
assert 'en' in voices and 'es' in voices, 'Faltan las voces de Piper: ejecuta antes la sección 4'
OUT_DIR = '/content/out'; os.makedirs(OUT_DIR, exist_ok=True)

def speak(text, lang='en'):
    """Texto -> ruta de un .wav con la voz de Piper de ese idioma."""
    return piper_wav(voices[lang], text, f'{OUT_DIR}/informe_{time.time_ns()}.wav')

## 8. Informe y voz en español
Mismos hechos, otro idioma: frases en español, nombres de objetos con un diccionario de las 80 clases COCO, y el traductor *opus-mt-en-es* solo para el caption de BLIP y, si la persona habló inglés, para su idea. Si esta parte falla, la app entrega igualmente el informe en inglés.

In [10]:
from transformers import MarianMTModel, MarianTokenizer

# ---------- traductor inglés -> español (Marian, solo para el texto de los modelos) ----------
MT_ID    = 'Helsinki-NLP/opus-mt-en-es'
mt_tok   = MarianTokenizer.from_pretrained(MT_ID, cache_dir=MODEL_CACHE)
mt_model = MarianMTModel.from_pretrained(MT_ID, cache_dir=MODEL_CACHE).to('cuda').eval()
vram('tras traductor')

def translate_es(text, max_new_tokens=96):
    """Texto corto en inglés -> español."""
    if not text.strip():
        return ''
    inp = mt_tok([text], return_tensors='pt', padding=True, truncation=True, max_length=128).to('cuda')
    with torch.no_grad():
        out = mt_model.generate(**inp, max_new_tokens=max_new_tokens, num_beams=4)
    return mt_tok.decode(out[0], skip_special_tokens=True)

# ---------- Etapa 5: informe en español ----------
COCO_ES = {
    'person': 'persona', 'bicycle': 'bicicleta', 'car': 'auto', 'motorcycle': 'motocicleta', 'airplane': 'avión',
    'bus': 'autobús', 'train': 'tren', 'truck': 'camión', 'boat': 'barco', 'traffic light': 'semáforo',
    'fire hydrant': 'hidrante', 'stop sign': 'señal de alto', 'parking meter': 'parquímetro', 'bench': 'banca',
    'bird': 'pájaro', 'cat': 'gato', 'dog': 'perro', 'horse': 'caballo', 'sheep': 'oveja', 'cow': 'vaca',
    'elephant': 'elefante', 'bear': 'oso', 'zebra': 'cebra', 'giraffe': 'jirafa', 'backpack': 'mochila',
    'umbrella': 'paraguas', 'handbag': 'bolso', 'tie': 'corbata', 'suitcase': 'maleta', 'frisbee': 'frisbi',
    'skis': 'esquís', 'snowboard': 'tabla de snowboard', 'sports ball': 'pelota', 'kite': 'papalote',
    'baseball bat': 'bate de béisbol', 'baseball glove': 'guante de béisbol', 'skateboard': 'patineta',
    'surfboard': 'tabla de surf', 'tennis racket': 'raqueta de tenis', 'bottle': 'botella',
    'wine glass': 'copa de vino', 'cup': 'taza', 'fork': 'tenedor', 'knife': 'cuchillo', 'spoon': 'cuchara',
    'bowl': 'tazón', 'banana': 'plátano', 'apple': 'manzana', 'sandwich': 'sándwich', 'orange': 'naranja',
    'broccoli': 'brócoli', 'carrot': 'zanahoria', 'hot dog': 'hot dog', 'pizza': 'pizza', 'donut': 'dona',
    'cake': 'pastel', 'chair': 'silla', 'couch': 'sofá', 'potted plant': 'planta en maceta', 'bed': 'cama',
    'dining table': 'mesa', 'toilet': 'inodoro', 'tv': 'televisión', 'laptop': 'laptop', 'mouse': 'ratón',
    'remote': 'control remoto', 'keyboard': 'teclado', 'cell phone': 'celular', 'microwave': 'microondas',
    'oven': 'horno', 'toaster': 'tostador', 'sink': 'lavabo', 'refrigerator': 'refrigerador', 'book': 'libro',
    'clock': 'reloj', 'vase': 'florero', 'scissors': 'tijeras', 'teddy bear': 'oso de peluche',
    'hair drier': 'secadora de pelo', 'toothbrush': 'cepillo de dientes',
}

VERDICT_ES = {
    'coincide':              'La imagen generada coincide con tu idea.',
    'coincide parcialmente': 'La imagen generada coincide parcialmente con tu idea.',
    'dudoso':                'No es seguro que la imagen generada coincida con tu idea.',
    'no coincide':           'La imagen generada no coincide con tu idea.',
}

def _conj(word):
    """'y' se vuelve 'e' ante una palabra que empieza con sonido 'i' (inodoro, hidrante), no ante 'hie'/'ia'..."""
    w = word.lower()
    return 'e' if (w.startswith('i') and not w[1:2] in ('a', 'e', 'o', 'u')) or \
                  (w.startswith('hi') and not w[2:3] in ('a', 'e', 'o', 'u')) else 'y'

def join_es(items):
    items = list(items)
    return items[0] if len(items) == 1 else ', '.join(items[:-1]) + f' {_conj(items[-1])} ' + items[-1]

def _lc(s):
    """Primera letra en minúscula, salvo que la segunda sea mayúscula (siglas)."""
    return s if (len(s) > 1 and s[1].isupper()) else s[:1].lower() + s[1:]

def report_es(f, heard_es, caption_es):
    """Facts (sin idioma) + textos ya en español -> informe en español."""
    es = lambda xs: [COCO_ES.get(c, c) for c in xs]
    s = [f'Entendí: {heard_es.strip().rstrip(".")}.', VERDICT_ES[f.verdict]]
    if f.requested:
        if not f.missing:
            s.append(f'El detector encontró todo lo que pediste: {join_es(es(f.found))}.')
        elif f.found:
            s.append(f'El detector encontró {join_es(es(f.found))}, pero no {join_es(es(f.missing))}.')
        else:
            s.append(f'El detector no encontró {join_es(es(f.missing))}.')
    for a, b in f.doubts:
        s.append(f'Dudó entre {join_es(es([a, b]))}.')
    if caption_es:
        s.append(f'El modelo de visión la describe así: {_lc(caption_es.strip().rstrip("."))}.')
    if f.texts:
        s.append(f'Texto leído en la imagen: {join_es(f.texts)}.')
    return ' '.join(s)

_bad = sorted(set(yolo.names.values()) ^ set(COCO_ES))          # mismo seguro contra typos que en L12
assert not _bad, f'COCO_ES no coincide con las clases de YOLO: {_bad}'

def spanish_version(f, idea):
    """Facts + idea -> (informe en español, ruta del .wav con su voz)."""
    heard_es   = idea.heard.text if idea.heard.lang == 'es' else translate_es(idea.prompt_en)
    caption_es = translate_es(f.caption) if f.caption else ''
    text = report_es(f, heard_es, caption_es)
    return text, speak(text, 'es')

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  312MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

  VRAM  5.56 / 15.6 GB   tras traductor


## 9. La puerta única: `analyse_idea()`
Es el único punto de entrada de la app y nunca lanza excepciones. Valida el idioma, ejecuta las etapas en orden (voz → imagen → verificación → informes) reportando el progreso, y devuelve un `IdeaOutcome`. Distingue errores de entrada (mensaje al usuario), falta de memoria de GPU y fallos internos (el detalle va al registro).

In [11]:
import traceback

# ---------- voz -> idea (texto en el idioma hablado + prompt en inglés) ----------
class Idea(NamedTuple):
    heard: Heard
    prompt_en: str

def translate_to_en(y, lang):
    """Whisper con task='translate' (L19, modo 4): audio en español -> texto en inglés."""
    res  = asr.transcribe(y, task='translate', language=lang, fp16=True)
    segs = [s for s in res['segments'] if s.get('no_speech_prob', 0) <= NO_SPEECH_MAX]   # mismo filtro que transcribe_idea
    text = ' '.join(s['text'].strip() for s in segs).strip()
    if not text:
        raise EntradaInvalida('No pude traducir tu idea al inglés para dibujarla. Intenta de nuevo.')
    return text

def understand(x, language=None):
    """Audio -> Idea(lo que dijo, prompt en inglés para Stable Diffusion)."""
    y = coerce_audio(x)
    text, lang, p = transcribe_idea(y, language)
    heard = Heard(text, lang, p, len(y) / SR)
    return Idea(heard, text if lang == 'en' else translate_to_en(y, lang))

# ---------- resultado y puerta única ----------
LIMIT_NOTE = 'La verificación comprueba que la escena corresponda a tu idea; no garantiza colores ni detalles finos.'
AI_NOTE    = 'La imagen y las descripciones las generan modelos de IA y pueden equivocarse.'

class IdeaOutcome(NamedTuple):
    error: str = ''
    heard: str = ''          # lo que Whisper entendió, en el idioma original
    lang: str = ''
    prompt_en: str = ''      # lo que recibió Stable Diffusion
    image: object = None     # la imagen generada
    boxes: object = None     # la misma imagen con las cajas de YOLO
    rows: list = []          # [[objeto, confianza, ruta, resultado, detalle], ...]
    verdict: str = ''
    clip: float = 0.0
    report: str = ''         # informe en inglés
    audio: str = ''          # voz del informe en inglés
    notes: list = []
    seed: int = 0
    timings: str = ''
    report_es: str = ''      # informe en español
    audio_es: str = ''       # voz del informe en español

def _noop(frac, desc):
    pass

def analyse_idea(x, language=None, progress=_noop):
    """ÚNICA puerta de entrada de la app. Nunca lanza excepción: siempre devuelve un IdeaOutcome."""
    try:
        if language not in (None, 'es', 'en'):
            raise EntradaInvalida('Idioma no válido. Elige Automático, Español o English.')
        T = {}
        t0 = time.time(); progress(0.05, 'Escuchando tu idea…')
        idea = understand(x, language);                    T['voz→texto'] = time.time() - t0
        t0 = time.time(); progress(0.25, 'Dibujando tu idea (unos 4 s)…')
        pic = draw(idea.prompt_en);                        T['imagen'] = time.time() - t0
        t0 = time.time(); progress(0.65, 'Verificando la imagen…')
        v = verify(idea.prompt_en, pic.image);             T['verificación'] = time.time() - t0

        t0 = time.time(); progress(0.80, 'Preparando el informe y su voz en inglés…')
        f = facts_from(idea.prompt_en, v)                  # los hechos, sin idioma: de aquí salen los dos informes
        report = report_en(f)
        wav = speak(report, 'en');                         T['informe EN'] = time.time() - t0

        t0 = time.time(); progress(0.92, 'Preparando el informe y su voz en español…')
        report_es_txt, wav_es, es_failed = '', '', False
        try:                                               # si el español falla, el informe en inglés se entrega igual
            report_es_txt, wav_es = spanish_version(f, idea)
        except Exception:
            traceback.print_exc()
            es_failed = True
        T['informe ES'] = time.time() - t0
        progress(1.0, 'Listo')

        notes = []
        if idea.heard.lang != 'en':
            notes.append(f'Se tradujo tu idea al inglés para dibujarla: «{idea.prompt_en}».')
        if es_failed:
            notes.append('No se pudo generar el informe en español; se muestra solo el de inglés.')
        else:
            notes.append('El informe en español usa un traductor automático (opus-mt) para el texto de los modelos y puede tener errores.')
        if language is None and idea.heard.lang_p < 0.8:
            notes.append(f'Detecté el idioma ({idea.heard.lang}) con poca seguridad ({idea.heard.lang_p:.0%}); si falló, elígelo a mano.')
        if pic.truncated:
            notes.append('Tu descripción era muy larga: el modelo de imagen solo leyó el principio.')
        notes += [LIMIT_NOTE, AI_NOTE]

        rows = [[r.task.label, f'{r.task.conf:.2f}', r.task.route, r.output or '—', r.info] for r in v.results]
        timings = ' · '.join(f'{k} {s:.1f} s' for k, s in T.items())
        return IdeaOutcome('', idea.heard.text, idea.heard.lang, idea.prompt_en, pic.image, v.boxes,
                           rows, v.verdict, v.clip, report, wav, notes, pic.seed, timings,
                           report_es=report_es_txt, audio_es=wav_es)

    except EntradaInvalida as e:                 # culpa de la entrada: el mensaje es para el usuario
        return IdeaOutcome(error=str(e))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return IdeaOutcome(error='La GPU se quedó sin memoria. Intenta de nuevo con una descripción más corta.')
    except Exception:                            # culpa del código: detalle al log, mensaje genérico
        traceback.print_exc()
        return IdeaOutcome(error='Ocurrió un error interno. Intenta de nuevo.')

## 10. Interfaz Gradio
Graba o sube un audio (1–15 s), elige el idioma (o automático) y pulsa **Crear imagen**. Muestra barra de progreso, la imagen, el veredicto, los dos informes con su voz, qué hizo cada modelo con cada objeto y avisos. La X del audio limpia todo. Hay dos ejemplos por si falla el micrófono, que solo funciona abriendo el enlace público en una pestaña nueva.

In [12]:
import gradio as gr
import pandas as pd

LANG_MAP = {'Automático': None, 'Español': 'es', 'English': 'en'}
COLUMNS  = ['Objeto', 'Confianza', 'Ruta', 'Resultado', 'Detalle / tiempo']
VERDICT_MD = {
    'coincide':              '✅ **La imagen coincide con tu idea**',
    'coincide parcialmente': '🟡 **La imagen coincide parcialmente con tu idea**',
    'dudoso':                '🟠 **No es seguro que la imagen coincida con tu idea**',
    'no coincide':           '❌ **La imagen no coincide con tu idea**',
}

def gradio_idea(audio, idioma, progress=gr.Progress()):
    o = analyse_idea(audio, LANG_MAP[idioma], progress=lambda f, d: progress(f, desc=d))
    if o.error:
        return None, None, f'### ❌ No pude procesar tu idea\n**{o.error}**', '', None, '', None, pd.DataFrame(columns=COLUMNS), ''
    status = (f'{VERDICT_MD[o.verdict]}  \nCLIP = {o.clip:.2f}\n\n'
              f'**Entendí ({o.lang}):** {o.heard}  \n**Prompt para la imagen:** {o.prompt_en}  \n'
              f'<sub>semilla {o.seed} · {o.timings}</sub>')
    notes = '### ℹ️ Avisos\n' + '\n'.join(f'- {n}' for n in o.notes)
    return (o.image, o.boxes, status, o.report_es, o.audio_es or None, o.report, o.audio,
            pd.DataFrame(o.rows, columns=COLUMNS), notes)

with gr.Blocks(title='Transmisor de ideas') as demo:
    gr.Markdown(
        '# 🎙️➡️🖼️ Transmisor de ideas\n'
        '**Describe una imagen con tu voz** y la app la dibuja, comprueba si se parece a lo que pediste y te lo cuenta en voz alta, en español y en inglés.\n\n'
        '1. Graba tu idea (de 1 a 15 s) o sube un audio. 2. Pulsa **Crear imagen**. 3. Mira la imagen y escucha el informe.\n\n'
        '*Modelos: Whisper (voz→texto) · Stable Diffusion (imagen) · YOLOv8 + BLIP + OCR + CLIP (verificación) · opus-mt (traducción) · Piper (voz del informe).*\n\n'
        '<sub>¿Usas Brave y la grabación sale sin sonido? Desactiva los Shields para esta página o prueba con otro navegador.</sub>')
    with gr.Row():
        with gr.Column(scale=1):
            in_audio = gr.Audio(sources=['microphone', 'upload'], type='filepath',
                                label='Tu idea (1–15 s, en español o inglés)',
                                editable=False,    # sin herramienta de recorte: nadie recorta el audio sin querer
                                waveform_options=gr.WaveformOptions(show_recording_waveform=False))   # la página hace menos trabajo al grabar
            in_lang  = gr.Radio(list(LANG_MAP), value='Automático', label='Idioma que hablas')
            btn      = gr.Button('🎨 Crear imagen', variant='primary')
            out_status = gr.Markdown()
        with gr.Column(scale=1):
            out_image = gr.Image(label='Imagen generada', type='pil')
    with gr.Row():
        with gr.Column():
            gr.Markdown('### Informe en español')
            out_report_es = gr.Textbox(label='Informe', lines=6, interactive=False)
            out_audio_es  = gr.Audio(label='Informe en voz (español)', type='filepath')
        with gr.Column():
            gr.Markdown('### Report in English')
            out_report_en = gr.Textbox(label='Report', lines=6, interactive=False)
            out_audio_en  = gr.Audio(label='Report audio (English)', type='filepath')
    with gr.Row():
        out_boxes = gr.Image(label='Lo que vio el verificador (cajas de YOLO)', type='pil', scale=1)
        out_table = gr.Dataframe(label='Qué hizo cada modelo con cada objeto', interactive=False, wrap=True, scale=2)
    out_notes = gr.Markdown()

    OUTS = [out_image, out_boxes, out_status, out_report_es, out_audio_es, out_report_en, out_audio_en, out_table, out_notes]
    btn.click(fn=gradio_idea, inputs=[in_audio, in_lang], outputs=OUTS)

    def limpiar():
        """Se ejecuta al pulsar la X del audio: deja todas las salidas como al inicio."""
        return None, None, '', '', None, '', None, pd.DataFrame(columns=COLUMNS), ''
    in_audio.clear(fn=limpiar, inputs=None, outputs=OUTS)

    # Respaldo si el micrófono falla: un clic carga el audio y el idioma; solo falta pulsar «Crear imagen»
    gr.Examples(examples=[[clips['es_valido'], 'Español'], [clips['en_valido'], 'English']],
                inputs=[in_audio, in_lang], label='Ejemplos (sin micrófono)')

gr.close_all()
demo.queue().launch(theme=gr.themes.Soft(), share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f552a4e32ac437aac3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 11. Detener la demo
Pon `STOP_DEMO = True` y ejecuta la celda.

In [15]:
STOP_DEMO = False   # cámbialo a True y ejecuta esta celda para detener la demo

if STOP_DEMO:
    demo.close()
    print('Demo detenida.')
else:
    print('La demo sigue activa. Pon STOP_DEMO = True para detenerla.')

Closing server running on port: 7860
Demo detenida.


## 12. Versiones instaladas

In [14]:
from importlib.metadata import version
for p in ['ultralytics', 'transformers', 'accelerate', 'gradio', 'pytesseract',
          'openai-whisper', 'piper-tts', 'diffusers', 'sentencepiece']:
    print(f'{p}=={version(p)}')
print('torch', torch.__version__)

ultralytics==8.4.170
transformers==5.16.1
accelerate==1.14.0
gradio==6.26.0
pytesseract==0.3.13
openai-whisper==20250625
piper-tts==1.8.0
diffusers==0.40.0
sentencepiece==0.2.2
torch 2.11.0+cu128
